Symbolic AI Concepts (Part 3)
## Symbolic Reasoning Foundations for Neural Researchers

This notebook is for people who are strong in **deep learning** but want the minimum set of **symbolic reasoning** skills needed for Neuro‑Symbolic AI.

### What this notebook covers
- Knowledge representation (facts, rules, ontologies)
- Closed‑world vs open‑world assumptions
- CSPs
- SAT & SMT (conceptual, with tiny examples)
- Determinism, soundness, completeness (engineering view)

## Setup

We’ll use mostly the Python standard library.

For SMT (Z3), there is an optional snippet. If you don’t have it installed, the notebook will still run.  
If you want Z3: `pip install z3-solver`


In [ ]:
from itertools import product
from dataclasses import dataclass

def hr(title: str):
    print("\n" + "="*80)
    print(title)
    print("="*80)


## 1) Knowledge Representation: Facts, Rules, Ontologies

A practical neuro‑symbolic pipeline often separates:

- **Facts**: observed truths (from logs, DBs, sensors, LLM extraction, etc.)
- **Rules**: domain logic (compliance policies, workflow constraints, business logic)
- **Ontology / schema**: the allowed vocabulary (types, relations, constraints)

Why ontologies/schemas matter in enterprise:
- prevent “nonsense” facts from entering the system
- enable deterministic validation
- reduce LLM hallucination impact when tools are schema‑driven

Below: validate triples against a tiny schema, then derive a new fact via a rule.


In [ ]:
hr("Schema validation + rule-based derivation")

# A few typed entities
types = {
    "alice": "User",
    "svc_payments": "Service",
    "db_prod": "Database",
    "kafka_1": "Queue",
}

# Allowed relations with (domain_type, range_type)
schema = {
    "owns": ("User", "Service"),
    "depends_on": ("Service", "Database"),
    "publishes_to": ("Service", "Queue"),
}

# Facts in triple form (subject, relation, object)
triples = {
    ("alice", "owns", "svc_payments"),
    ("svc_payments", "depends_on", "db_prod"),
    ("svc_payments", "publishes_to", "kafka_1"),
    ("db_prod", "owns", "alice"),  # ❌ nonsense: domain/range mismatch
}

def validate(triples):
    violations = []
    for s, r, o in triples:
        if r not in schema:
            violations.append((s, r, o, "unknown_relation"))
            continue
        dom, rng = schema[r]
        if types.get(s) != dom:
            violations.append((s, r, o, f"domain_expected_{dom}"))
        if types.get(o) != rng:
            violations.append((s, r, o, f"range_expected_{rng}"))
    return violations

viol = validate(triples)
print("Violations:")
for v in viol:
    print(" ", v)

# Now a rule: depends_on(Service, Database) -> needs_backup(Database)
needs_backup = set()
for (s, r, o) in triples:
    if r == "depends_on":
        needs_backup.add(o)

print("\nDerived needs_backup facts:", needs_backup)


## 2) Closed‑World vs Open‑World Assumptions

This is a major conceptual difference that affects how agents behave.

### Closed‑World Assumption (CWA)
“If it’s not in the knowledge base, assume it’s false.”
- Common in databases, Datalog, many enterprise systems.
- Great for enforcement: missing permission ⇒ deny.

### Open‑World Assumption (OWA)
“If it’s not in the knowledge base, we don’t know.”
- Common in Semantic Web / OWL ontologies.
- Great when knowledge is incomplete and you must avoid false negatives.

Realistic example: authorization.

- Under CWA: if we do **not** know that user is allowed ⇒ deny.
- Under OWA: if we do **not** know ⇒ unknown (you might ask for more info).


In [ ]:
hr("CWA vs OWA: authorization check")

allowed = {("alice", "deploy_prod")}  # known allow-list

def can_do_closed_world(user, action):
    # Not explicitly allowed => deny
    return (user, action) in allowed

def can_do_open_world(user, action):
    # Not explicitly allowed => unknown (need more knowledge)
    return True if (user, action) in allowed else None

user, action = "bob", "deploy_prod"

print(f"Query: allowed({user}, {action})?")
print("Closed-world:", can_do_closed_world(user, action), "=> enforce deny if False")
print("Open-world:", can_do_open_world(user, action), "=> None means unknown; request evidence/approval")


## 3) Constraint Satisfaction Problems (CSPs)

A **CSP** consists of:
- variables
- domains (possible values)
- constraints

Realistic example: a tiny on-call schedule.

Constraints:
- Assign 1 person to each day.
- No one can be on call two days in a row.
- Some people are unavailable on certain days.

We’ll brute-force (small) just for intuition.


In [ ]:
hr("CSP: tiny on-call schedule")

days = ["Mon", "Tue", "Wed"]
people = ["alice", "bob", "carol"]

# Availability constraints
unavailable = {
    ("alice", "Tue"),
    ("carol", "Mon"),
}

def available(p, d):
    return (p, d) not in unavailable

def valid(assign):
    # assign: dict day -> person
    # availability
    for d, p in assign.items():
        if not available(p, d):
            return False
    # no consecutive days for same person
    for i in range(len(days)-1):
        if assign[days[i]] == assign[days[i+1]]:
            return False
    return True

solution = None
for vals in product(people, repeat=len(days)):
    assign = dict(zip(days, vals))
    if valid(assign):
        solution = assign
        break

print("One feasible schedule:", solution)


## 4) SAT (Boolean Satisfiability)

**SAT** asks:
> Is there a True/False assignment that satisfies all constraints?

Realistic example: feature-flag configuration.

Constraints:
- If `PROD` is true, then `EVAL_PASSED` must be true.
- If `PROD` is true, then `AUDIT_LOGS` must be true.
- If `DEBUG` is true, then `PROD` must be false. (no debug in prod)

We’ll brute force all assignments and find valid configs.


In [ ]:
hr("SAT: brute-force valid configurations")

vars_ = ["PROD", "EVAL_PASSED", "AUDIT_LOGS", "DEBUG"]

def implies(a, b): return (not a) or b

def constraints(a):
    return (
        implies(a["PROD"], a["EVAL_PASSED"]) and
        implies(a["PROD"], a["AUDIT_LOGS"]) and
        implies(a["DEBUG"], (not a["PROD"]))
    )

solutions = []
for vals in product([False, True], repeat=len(vars_)):
    a = dict(zip(vars_, vals))
    if constraints(a):
        solutions.append(a)

print(f"Found {len(solutions)} valid configurations.")
print("Example valid configs (first 5):")
for s in solutions[:5]:
    print(" ", s)

print("\nInterpretation: SAT solvers do this at huge scale without brute force.")


## 5) SMT (Satisfiability Modulo Theories)

**SMT** extends SAT beyond booleans to richer domains (integers, reals, etc.).

Realistic example: latency budgeting for an agentic workflow.

Constraints:
- total_latency = llm + tools + overhead
- must be <= SLA budget (e.g., 1200 ms)
- tools latency is at least some minimum

We’ll show an optional Z3 snippet that finds a feasible assignment.


In [ ]:
hr("SMT: optional Z3 example (latency budgeting)")

try:
    from z3 import Int, And, Solver

    llm = Int("llm_ms")
    tools = Int("tools_ms")
    overhead = Int("overhead_ms")

    budget = 1200

    s = Solver()
    # bounds
    s.add(And(llm >= 200, llm <= 900))
    s.add(And(tools >= 100, tools <= 800))
    s.add(And(overhead >= 50, overhead <= 200))

    # SLA constraint
    s.add(llm + tools + overhead <= budget)

    if s.check().r == 1:
        m = s.model()
        sol = {"llm_ms": int(str(m[llm])), "tools_ms": int(str(m[tools])), "overhead_ms": int(str(m[overhead]))}
        print("One feasible solution:", sol)
        print("Total:", sol["llm_ms"] + sol["tools_ms"] + sol["overhead_ms"], "ms")
    else:
        print("UNSAT: no feasible latency allocation")
except Exception as e:
    print("Z3 not installed (that's fine). To run this cell: pip install z3-solver")
    print("Error:", type(e).__name__)


## 6) Determinism, Soundness, Completeness (Engineering View)

These show up constantly when you embed reasoning into **real systems**.

### Determinism
Same input ⇒ same output.
- Helps debugging
- Helps reproducibility
- Helps “did the system change?” checks (regressions)

### Soundness
Everything you derive is **correct** given your rules and facts.
- Unsound rules can produce dangerous false conclusions.

### Completeness
If something is entailed by your rules/facts, you can derive it (eventually).
- Incomplete procedures may miss valid results (often due to timeouts, depth limits, pruning).

Below: a quick demonstration of each.


In [ ]:
hr("Soundness: a buggy rule deriving nonsense")

# Facts: Only Alice is a User
User = {"alice"}

# Buggy rule: User(x) -> IsDatabase(x)  (nonsense / unsound)
def buggy_derive_databases(users):
    IsDatabase = set()
    for x in users:
        IsDatabase.add(x)
    return IsDatabase

print("Derived IsDatabase facts (unsound):", buggy_derive_databases(User))

hr("Determinism: stable output vs randomness")

def deterministic(x):
    return x * 2

print("deterministic(21) twice:", deterministic(21), deterministic(21))

# Contrast: a nondeterministic function could differ due to sampling, race conditions, etc.
import random
def nondeterministic():
    return random.choice(["A", "B", "C"])

print("nondeterministic() samples:", [nondeterministic() for _ in range(5)])

hr("Completeness: depth-limited proof can miss a true entailment")

Edge = {("A","B"), ("B","C"), ("C","D")}

def can_reach_depth_limited(start, target, depth):
    if start == target:
        return True
    if depth == 0:
        return False
    for (u, v) in Edge:
        if u == start:
            if can_reach_depth_limited(v, target, depth-1):
                return True
    return False

print("True fact: Reachable(A, D) is entailed by edges.")
print("But with depth=1:", can_reach_depth_limited("A", "D", 1), "(missed => incomplete)")
print("With depth=3:", can_reach_depth_limited("A", "D", 3), "(found)")
